# MT5 AI Trader — Colab Train + Walk-Forward

Notebook này dùng Colab để clone repo, đọc CSV từ Google Drive, train model, chạy walk-forward, rồi lưu `models/` và `reports/` về Drive.

> Colab không chạy MT5 terminal. MT5/paper/demo/live vẫn chạy local Windows.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf mt5-ai-trader
!git clone https://github.com/quoccuongphan300992-cmd/mt5-ai-trader.git
%cd /content/mt5-ai-trader
!git checkout main
!git pull --ff-only origin main
!git log -1 --oneline
!ls -lh


In [ ]:
!python --version
!nproc
!free -h
!df -h


In [ ]:
!pip install -r requirements-colab.txt


## Cấu hình đường dẫn CSV

Upload CSV EURUSD H1 vào Google Drive, ví dụ:

`MyDrive/mt5-ai/data/EURUSD_H1.csv`

Nếu file tên khác, sửa biến `DRIVE_CSV` bên dưới.


In [ ]:
DRIVE_CSV = "/content/drive/MyDrive/mt5-ai/data/EURUSD_H1.csv"
LOCAL_CSV = "data/EURUSD_H1.csv"
SYMBOL = "EURUSD"
TIMEFRAME = "H1"
BARS = 50000


In [ ]:
!ls -lh "/content/drive/MyDrive/mt5-ai/data" || true


In [ ]:
import os, shutil
os.makedirs("data", exist_ok=True)
if not os.path.exists(DRIVE_CSV):
    raise FileNotFoundError(f"CSV not found: {DRIVE_CSV}. Upload CSV to Drive or edit DRIVE_CSV.")
shutil.copy2(DRIVE_CSV, LOCAL_CSV)
print("CSV copied:", LOCAL_CSV, os.path.getsize(LOCAL_CSV), "bytes")


In [ ]:
!python main.py train --csv "$LOCAL_CSV" --symbol "$SYMBOL" --timeframe "$TIMEFRAME" --bars "$BARS" --label-method atr_path --label-atr-tp-mult 1.5 --label-atr-sl-mult 1.0 --model-type extra_trees
!ls -lh models


In [ ]:
import json
meta = json.load(open('models/metadata.json'))
print('feature_count:', meta.get('feature_count'))
print('has new features:', all(f in meta.get('features', []) for f in ['adx_14', 'atr_percentile_100', 'spread_percentile_100', 'trend_stack_bear']))
print([f for f in ['adx_14', 'atr_percentile_100', 'spread_percentile_100', 'spread_to_atr', 'is_london_session', 'trend_stack_bear'] if f in meta.get('features', [])])


In [ ]:
!python main.py walk-forward --csv "$LOCAL_CSV" --symbol "$SYMBOL" --timeframe "$TIMEFRAME" --bars "$BARS" --direction SELL --min 0.48 --max 0.68 --step 0.02 --label-method atr_path --label-atr-tp-mult 1.5 --label-atr-sl-mult 1.0 --model-type extra_trees
!python main.py walk-forward --csv "$LOCAL_CSV" --symbol "$SYMBOL" --timeframe "$TIMEFRAME" --bars "$BARS" --direction SELL --threshold 0.54 --label-method atr_path --label-atr-tp-mult 1.5 --label-atr-sl-mult 1.0 --model-type extra_trees --min-atr-percentile 0.1 --max-atr-percentile 0.95 --max-spread-percentile 0.9
!ls -lh reports


In [ ]:
import pandas as pd
summary = pd.read_csv("reports/walk_forward_summary.csv")
cols = ["threshold", "candidate_pass", "total_trades", "positive_expectancy_folds", "positive_pf_folds", "overall_profit_factor", "overall_expectancy_r", "max_fold_drawdown_pct"]
display(summary[[c for c in cols if c in summary.columns]])


In [ ]:
!mkdir -p "/content/drive/MyDrive/mt5-ai/outputs/models"
!mkdir -p "/content/drive/MyDrive/mt5-ai/outputs/reports"
!cp -r models/* "/content/drive/MyDrive/mt5-ai/outputs/models/" 2>/dev/null || true
!cp -r reports/* "/content/drive/MyDrive/mt5-ai/outputs/reports/" 2>/dev/null || true
!zip -r mt5_ai_outputs.zip models reports
!cp mt5_ai_outputs.zip "/content/drive/MyDrive/mt5-ai/outputs/mt5_ai_outputs.zip"
!find "/content/drive/MyDrive/mt5-ai/outputs" -maxdepth 3 -type f -print


## Optional: Auto-Improve Offline Search

This searches candidate model/label/filter/threshold configs using walk-forward validation.

It is offline only. It does not change MT5, demo, paper, or live execution behavior.

Production model artifacts are written only if a candidate passes validation.


In [ ]:
!python main.py auto-improve \
  --csv "$LOCAL_CSV" \
  --symbol "$SYMBOL" \
  --timeframe "$TIMEFRAME" \
  --bars "$BARS" \
  --max-rounds 30


In [ ]:
!cat reports/auto_improve_best.json


In [ ]:
import pandas as pd
df = pd.read_csv("reports/auto_improve_candidates.csv")
print(df.head(20).to_string())
